In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px

zema = ZemaManager()


In [0]:
url = "https://ldcom365.sharepoint.com"

In [0]:
start_date = datetime(2020, 1, 1)

end_date = datetime(2030, 1, 1)

In [0]:
eurusd= zema.get_curve(curve='FX_GPL_FWD_EURUSD', period=f"{start_date}::{end_date}")

import calendar

def recreate_full_date(contract_month, contract_year):
    # Define the base date and base number
    base_date = datetime(2024, 12, 2)
    base_number = 4336

    # Calculate the number of days offset
    delta_days = contract_month - base_number
    calculated_date = base_date + timedelta(days=delta_days)

    # Extract the day of the month and month number
    day_of_month = calculated_date.day
    month_number = calculated_date.month

    # Get the last day of the target month
    last_day_of_month = calendar.monthrange(contract_year, month_number)[1]

    # Adjust the day if it exceeds the last day of the month
    if day_of_month > last_day_of_month:
        day_of_month = last_day_of_month

    # Use the provided year from the contract_year column
    return datetime(contract_year, month_number, day_of_month)
  

# Define a base date for the mapping
base_date = datetime(2024, 12, 2)
base_number = 4336

# Apply the mapping function to the column
eurusd['Contract Month Date'] = eurusd.apply(
    lambda row: recreate_full_date(row['contract_month'], row['contract_year']), axis=1
)

#eurusd=eurusd[['date','value','contract_year','Contract Month Date']]
eurusd['contract_month']=eurusd['Contract Month Date'].dt.month
eurusd.rename(columns={'value': 'eurusd'}, inplace=True)

# Group by the required columns and calculate the average eurusd
eurusd = eurusd.groupby(['date', 'contract_year', 'contract_month'], as_index=False)['eurusd'].mean()

eurusd

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/FX/EURUSD.xlsx',eurusd,index=False)


https://ldcom365.sharepoint.com/:x:/r/sites/GRP-TradingLineups/Shared%20Documents/PROVIDERS/BEANS/Beans_lineups.xlsx?d=w975ba290228442f6ac8fd6fc00677f2b&csf=1&web=1&e=ix1xGP

In [0]:
def assign_season_barley(row):
    date = row['date']
    if date.month >= 11:  # March to December → same year
        season_start = date.year
    else:  # January, February → previous year's marketing season
        season_start = date.year - 1
    return f"{season_start}/{season_start + 1}"

def adjust_virtual_date(row):
    if row['virtual_date'].month in [11, 12]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year - 1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date']

In [0]:

matif= zema.get_curve(curve="P-FUTURE-ENXT-INPUT-WHEAT-EUR-MT", period=f"{start_date}::{end_date}")

matif=matif[matif['observation']=='Settle']
matif.rename(columns={'value': 'matif_eur'}, inplace=True)
matif=matif[['date','matif_eur','contract_year','contract_month']]

merged_df = pd.merge(
    matif,
    eurusd,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'  # You can also use 'outer', 'left', or 'right' depending on your needs
)
merged_df['matif_usd']=merged_df['matif_eur']*merged_df['eurusd']

merged_df=merged_df[['date','matif_usd','contract_year','contract_month']]

matif_raw=merged_df.copy()
# Make sure date is datetime

matif_raw['date'] = pd.to_datetime(matif_raw['date'])

# Create a reference year column based on the date
matif_raw['ref_contract_year'] = matif_raw['date'].apply(lambda x: x.year + 1 if x.month >= 3 else x.year)

# Filter to March contracts matching the reference year
matif_march = matif_raw[
    (matif_raw['contract_month'] == 3) &
    (matif_raw['contract_year'] == matif_raw['ref_contract_year'])
].copy()

# Drop the helper column if needed
matif_march.drop(columns='ref_contract_year', inplace=True)

matif_march['date'] = pd.to_datetime(matif_march['date'])

# # Determine the correct contract year for each date
# matif_march['target_contract_year'] = matif_march['date'].dt.year
# matif_march.loc[matif_march['date'].dt.month >= 3, 'target_contract_year'] += 1

# # Filter to only keep rows where contract_year == target_contract_year
# matif_march_closest_contract = matif_march[matif_march['contract_year'] == matif_march['target_contract_year']].copy()


fob_barley = zema.get_curve(curve='T-CASH-LDC-RISK-FLAT-BARLEY-AR-FOB Bahia Blanca-USD-MT', period=f"{start_date}::{end_date}")
fob_barley=fob_barley[['date','value']]
barley_march_matif=pd.merge(fob_barley,matif_march,on='date')
barley_march_matif['Premium']=barley_march_matif['value']-barley_march_matif['matif_usd']


barley_march_matif['season']= barley_march_matif.apply(assign_season_barley, axis=1)
barley_march_matif['date'] = pd.to_datetime(barley_march_matif['date'])
barley_march_matif['virtual_date'] = pd.to_datetime({'year': 2000, 'month': barley_march_matif['date'].dt.month, 'day': barley_march_matif['date'].dt.day},errors='coerce')
# barley_march_matif['virtual_date'] = barley_march_matif.apply(adjust_virtual_date, axis=1)

In [0]:
barley_march_matif['virtual_date'] = barley_march_matif.apply(adjust_virtual_date, axis=1)
barley_march_matif


In [0]:
# Plot with Plotly
prem_barley_matif = px.line(
    barley_march_matif,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - MARCH MATIF'
)

prem_barley_matif.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
prem_barley_matif.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

prem_barley_matif.show()

prem_barley_matif_html = prem_barley_matif.to_html(include_plotlyjs='cdn', full_html=True)


### MATIF_DEC

In [0]:
# matif_raw['ref_contract_year'] = matif_raw['date'].apply(lambda x: x.year + 1 if x.month >= 12 else x.year)
matif_raw['ref_contract_year'] = matif_raw['date'].apply(lambda x: x.year)

# Filter to March contracts matching the reference year
matif_dec = matif_raw[
    (matif_raw['contract_month'] == 12) &
    (matif_raw['contract_year'] == matif_raw['ref_contract_year'])
].copy()

# Drop the helper column if needed
matif_dec.drop(columns='ref_contract_year', inplace=True)

matif_dec['date'] = pd.to_datetime(matif_dec['date'])

fob_barley = zema.get_curve(curve='T-CASH-LDC-RISK-FLAT-BARLEY-AR-FOB Bahia Blanca-USD-MT', period=f"{start_date}::{end_date}")
fob_barley=fob_barley[['date','value']]
barley_dec_matif=pd.merge(fob_barley,matif_dec,on='date')

barley_dec_matif['Premium']=barley_dec_matif['value']-barley_dec_matif['matif_usd']


barley_dec_matif['season']= barley_dec_matif.apply(assign_season_barley, axis=1)
barley_dec_matif['date'] = pd.to_datetime(barley_dec_matif['date'])
barley_dec_matif['virtual_date'] = pd.to_datetime({'year': 2000, 'month': barley_dec_matif['date'].dt.month, 'day': barley_dec_matif['date'].dt.day},errors='coerce')
barley_dec_matif['virtual_date'] = barley_dec_matif.apply(adjust_virtual_date, axis=1)

In [0]:
# Plot with Plotly
prem_barley_matif_dec = px.line(
    barley_dec_matif,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - DEC MATIF'
)

prem_barley_matif_dec.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
prem_barley_matif_dec.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

prem_barley_matif_dec.show()

prem_barley_matif_dec_html = prem_barley_matif_dec.to_html(include_plotlyjs='cdn', full_html=True)


### WHEAT UPR

In [0]:
wheat = zema.get_curve(curve="P-CASH-LDC-INPUT-FLAT-WHEAT-AR-FOB Up River-11.5-USD-MT", period=f"{start_date}::{end_date}")
wheat=wheat[wheat['observation']=='Last']
wheat=wheat[['date','value','contract_year','contract_month']]
wheat['day']=wheat['date'].dt.day
wheat['month']=wheat['date'].dt.month
wheat['year']=wheat['date'].dt.year
wheat['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': wheat['month'], 'day': wheat['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
wheat = wheat.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
wheat = wheat.sort_values(by='virtual_date')
wheat=wheat[['date','value','contract_year','contract_month','virtual_date']]

wheat_spot=wheat.copy()
# Ensure datetime format
wheat_spot['date'] = pd.to_datetime(wheat_spot['date'])

# Extract current month and year from the 'date' column
wheat_spot['ref_month'] = wheat_spot['date'].dt.month
wheat_spot['ref_year'] = wheat_spot['date'].dt.year

# Filter to potential spot contracts
spot_candidates = wheat_spot[
    (wheat_spot['contract_year'] == wheat_spot['ref_year']) &
    (wheat_spot['contract_month'] == wheat_spot['ref_month'])
].copy()

# Keep the first occurrence of the contract per date
spot_contracts = (
    spot_candidates
    .sort_values(['date'])  # Make sure earliest stays first
    .drop_duplicates(subset='date', keep='first')
)

# Optionally keep only relevant columns
UPR_Wheat= spot_contracts[['date', 'value']]
UPR_Wheat.rename(columns={'value': 'fob_upr'}, inplace=True)

fob_upr_wheat_barley=pd.merge(UPR_Wheat,fob_barley,on='date')
fob_upr_wheat_barley['Premium']=fob_upr_wheat_barley['value']-fob_upr_wheat_barley['fob_upr']

fob_upr_wheat_barley['season']= fob_upr_wheat_barley.apply(assign_season_barley, axis=1)
fob_upr_wheat_barley['date'] = pd.to_datetime(fob_upr_wheat_barley['date'])
fob_upr_wheat_barley['virtual_date'] = pd.to_datetime({'year': 2000, 'month': fob_upr_wheat_barley['date'].dt.month, 'day': fob_upr_wheat_barley['date'].dt.day},errors='coerce')
fob_upr_wheat_barley['virtual_date'] = fob_upr_wheat_barley.apply(adjust_virtual_date, axis=1)

# Plot with Plotly
upr_w_vs_barley = px.line(
    fob_upr_wheat_barley,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - FOB WHEAT UPR'
)

upr_w_vs_barley.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
upr_w_vs_barley.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

upr_w_vs_barley.show()

upr_w_vs_barley_html = upr_w_vs_barley.to_html(include_plotlyjs='cdn', full_html=True)



### CORN BB

In [0]:
corn_bb = zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-FOB Bahia Blanca Arg-USD-MT", period=f"{start_date}::{end_date}")
corn_bb=corn_bb[corn_bb['observation']=='Last']
corn_bb=corn_bb[['date','value','contract_year','contract_month']]
corn_bb['day']=corn_bb['date'].dt.day
corn_bb['month']=corn_bb['date'].dt.month
corn_bb['year']=corn_bb['date'].dt.year
corn_bb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': corn_bb['month'], 'day': corn_bb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)


# Drop rows with invalid virtual dates
corn_bb = corn_bb.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
corn_bb = corn_bb.sort_values(by='virtual_date')
corn_bb=corn_bb[['date','value','contract_year','contract_month','virtual_date']]

corn_bb_spot=corn_bb.copy()
# Ensure datetime format
corn_bb_spot['date'] = pd.to_datetime(corn_bb_spot['date'])

# Extract current month and year from the 'date' column
corn_bb_spot['ref_month'] = corn_bb_spot['date'].dt.month
corn_bb_spot['ref_year'] = corn_bb_spot['date'].dt.year

# Filter to potential spot contracts
spot_candidates = corn_bb_spot[
    (corn_bb_spot['contract_year'] == corn_bb_spot['ref_year']) &
    (corn_bb_spot['contract_month'] == corn_bb_spot['ref_month'])
].copy()

# Keep the first occurrence of the contract per date
spot_contracts = (
    spot_candidates
    .sort_values(['date'])  # Make sure earliest stays first
    .drop_duplicates(subset='date', keep='first')
)

# Optionally keep only relevant columns
corn_bb= spot_contracts[['date', 'value']]
corn_bb.rename(columns={'value': 'corn_bb'}, inplace=True)

fob_corn_bb_barley=pd.merge(corn_bb,fob_barley,on='date')
fob_corn_bb_barley['Premium']=fob_corn_bb_barley['value']-fob_corn_bb_barley['corn_bb']

fob_corn_bb_barley['season']= fob_corn_bb_barley.apply(assign_season_barley, axis=1)
fob_corn_bb_barley['date'] = pd.to_datetime(fob_corn_bb_barley['date'])
fob_corn_bb_barley['virtual_date'] = pd.to_datetime({'year': 2000, 'month': fob_corn_bb_barley['date'].dt.month, 'day': fob_corn_bb_barley['date'].dt.day},errors='coerce')
fob_corn_bb_barley['virtual_date'] = fob_corn_bb_barley.apply(adjust_virtual_date, axis=1)

# Plot with Plotly
BB_c_vs_barley = px.line(
    fob_corn_bb_barley,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - BB CORN UPR'
)

BB_c_vs_barley.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
BB_c_vs_barley.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

BB_c_vs_barley.show()

BB_c_vs_barley_html = BB_c_vs_barley.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
MATIF_VS_BARLEY_html_report  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Markets Snapshot</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>MARCH MATIF VS ARG FOB BARLEY</h1>

    <div class="chart-container">{prem_barley_matif_html}</div>

    <h1>DEC MATIF VS ARG FOB BARLEY</h1>

    <div class="chart-container">{prem_barley_matif_dec_html}</div>

    <h1>WHEAT UPR VS ARG FOB BARLEY</h1>

    <div class="chart-container">{upr_w_vs_barley_html}</div>

    <h1>CORN BB VS ARG FOB BARLEY</h1>

    <div class="chart-container">{BB_c_vs_barley_html}</div>


</body>
</html>
"""

MATIF_VS_BARLEY_report_bytes = MATIF_VS_BARLEY_html_report.encode("utf-8")

In [0]:
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Markets Snapshot</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        h1, h2 {{
            margin-top: 40px;
            color: #333;
        }}
        img {{
            width: 1800px;
            max-width: 100%;
            min-width: 1000px;
            display: block;
            margin-bottom: 30px;
        }}
    </style>
</head>
<body>
    <h1>MARCH MATIF VS FOB BARLEY</h1>

    <img src="mar_matif_barley" alt="Spot">

    <h1>DEC MATIF VS FOB BARLEY</h1>

    <img src="dec_matif_barley" alt="Spot">

    <h1>WHEAT UPR VS ARG FOB BARLEY</h1>

    <img src="wheat_upr" alt="Spot">

    <h1>CORN BB VS ARG FOB BARLEY</h1>

    <img src="corn_bb" alt="Spot">

</body>
</html>
"""

test=['florian.girardi-ext@ldc.com']

# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=test,
    subject=f'MATIF VS FOB BARLEY{datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    body=html_content,
    mime_type="html",
    html_images={"mar_matif_barley": prem_barley_matif,"dec_matif_barley":prem_barley_matif_dec,"wheat_upr":upr_w_vs_barley,"corn_bb":BB_c_vs_barley},
    attachment={"matif_vs_barley_fob.html": MATIF_VS_BARLEY_report_bytes}
)

